<a href="https://colab.research.google.com/github/Raffy0-1/DHC-Phase-2-ML-Task_4/blob/main/RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Chatbot
### Built with LangChain + FAISS + Google Gemini (Free API) + Streamlit

A conversational chatbot that:
- Reads Wikipedia articles as its knowledge base
- Remembers your last 5 messages (conversation memory)
- Finds the most relevant passages before answering (RAG)

| Component | Tool |
|---|---|
| LLM | Google Gemini 2.5 Flash |
| Embeddings | sentence-transformers (local) |
| Vector DB | FAISS (local) |
| UI | Streamlit + ngrok |
| Storage | Google Drive |

## Section 1 — Install Libraries

In [ ]:
# langchain              → core framework connecting LLMs, memory, retrievers
# langchain-community    → extra connectors: FAISS, Wikipedia, HuggingFace embeddings
# langchain-google-genai → LangChain adapter for Google Gemini
# langchain-classic      → houses create_retrieval_chain and related chain helpers
#                          (moved out of langchain core in v1.x)
# sentence-transformers  → local embedding model, no API key needed
# faiss-cpu              → vector similarity search
# wikipedia              → fetch Wikipedia articles as knowledge base
# streamlit              → web chat UI
# pyngrok                → tunnel to expose Streamlit publicly from Colab

!pip install -q \
    langchain \
    langchain-community \
    langchain-google-genai \
    langchain-classic \
    sentence-transformers \
    faiss-cpu \
    wikipedia \
    streamlit \
    pyngrok

print("All libraries installed successfully!")
print("If you see RED errors above (not warnings), restart runtime and rerun.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
All libraries installed successfully!
If you see RED errors above

---
## Section 2 — Mount Google Drive & Set Up Project Folder

Saving the vector store to Drive means subsequent sessions skip the rebuild step entirely.

```
MyDrive/
  └── rag_chatbot/
        ├── vectorstore/
        │     └── faiss_index/
        │           ├── index.faiss
        │           └── index.pkl
        └── app.py
```

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR      = "/content/drive/MyDrive/rag_chatbot"
VECTORSTORE_PATH = f"{PROJECT_DIR}/vectorstore/faiss_index"
APP_PATH         = f"{PROJECT_DIR}/app.py"

os.makedirs(os.path.dirname(VECTORSTORE_PATH), exist_ok=True)

print("Google Drive mounted!")
print(f"\nProject folder : {PROJECT_DIR}")
print(f"Vector store   : {VECTORSTORE_PATH}")
print(f"App file       : {APP_PATH}")

if os.path.exists(VECTORSTORE_PATH):
    print("\nSaved vector store found — you can skip Sections 3 & 4.")
else:
    print("\nNo saved vector store yet — run Sections 3 & 4 to build it.")

Mounted at /content/drive
Google Drive mounted!

Project folder : /content/drive/MyDrive/rag_chatbot
Vector store   : /content/drive/MyDrive/rag_chatbot/vectorstore/faiss_index
App file       : /content/drive/MyDrive/rag_chatbot/app.py

No saved vector store yet — run Sections 3 & 4 to build it.


---
## Section 3 — Load & Chunk Documents

> **Skip this section if you already have a saved vector store on Drive.**

A Wikipedia article can be 50,000+ words. Pasting the whole thing into every LLM prompt is too expensive and inefficient. Instead:

1. Split articles into ~500-character chunks
2. Convert each chunk into a vector (a semantic fingerprint)
3. At query time, fetch only the 3–5 most relevant chunks
4. Pass only those chunks to the LLM

A 50-character overlap between chunks preserves context at boundaries — so a sentence split across two chunks still makes sense in both.

In [ ]:
import wikipedia
from langchain_core.documents import Document

def load_wikipedia_pages(topics: list) -> list:
    """Fetch Wikipedia pages and return them as LangChain Document objects."""
    docs = []
    for topic in topics:
        print(f"Fetching: '{topic}'...", end=" ")
        try:
            page = wikipedia.page(topic, auto_suggest=False)
            docs.append(Document(
                page_content=page.content,
                metadata={"source": page.url, "title": page.title}
            ))
            print(f"({len(page.content):,} chars)")
        except wikipedia.DisambiguationError as e:
            print(f"ambiguous → using '{e.options[0]}'")
            page = wikipedia.page(e.options[0])
            docs.append(Document(
                page_content=page.content,
                metadata={"source": page.url, "title": page.title}
            ))
        except Exception as ex:
            print(f"FAILED: {ex}")

    total_chars = sum(len(d.page_content) for d in docs)
    print(f"\nLoaded {len(docs)} documents | {total_chars:,} total characters")
    return docs


TOPICS = [
    "Artificial intelligence",
    "Machine learning",
    "Natural language processing",
    "Transformer (machine learning model)",
    "Large language model",
    "Retrieval-augmented generation",
]

raw_docs = load_wikipedia_pages(TOPICS)

print(f"\nExample document:")
print(f"  Title  : {raw_docs[0].metadata['title']}")
print(f"  Source : {raw_docs[0].metadata['source']}")
print(f"  Length : {len(raw_docs[0].page_content):,} characters")
print(f"  Preview: {raw_docs[0].page_content[:300]}...")

Fetching: 'Artificial intelligence'... (84,281 chars)
Fetching: 'Machine learning'... (59,399 chars)
Fetching: 'Natural language processing'... (32,402 chars)
Fetching: 'Transformer (machine learning model)'... (113,468 chars)
Fetching: 'Large language model'... (63,499 chars)
Fetching: 'Retrieval-augmented generation'... (10,842 chars)

Loaded 6 documents | 363,891 total characters

Example document:
  Title  : Artificial intelligence
  Source : https://en.wikipedia.org/wiki/Artificial_intelligence
  Length : 84,281 characters
  Preview: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develo...


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter tries to split on paragraph breaks first,
# then line breaks, then sentences, then words — keeping chunks coherent.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(raw_docs)

print(f"Chunking complete!")
print(f"  Input  : {len(raw_docs)} documents")
print(f"  Output : {len(chunks)} chunks")
print(f"  Avg chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")

example_idx = 25
print(f"\nExample chunk #{example_idx}:")
print(f"  Source  : {chunks[example_idx].metadata['title']}")
print(f"  Length  : {len(chunks[example_idx].page_content)} chars")
print(f"  Content : {chunks[example_idx].page_content}")

Chunking complete!
  Input  : 6 documents
  Output : 1153 chunks
  Avg chunk size: 314 chars

Example chunk #25:
  Source  : Artificial intelligence
  Length  : 409 chars
  Content : In reinforcement learning, the agent is rewarded for good responses and punished for bad ones. The agent learns to choose responses that are classified as "good". Transfer learning is when the knowledge gained from one problem is applied to a new problem. Deep learning is a type of machine learning that runs inputs through biologically inspired artificial neural networks for all of these types of learning.


---
## Section 4 — Create Embeddings & Build Vector Store

> **Skip this section if you already have a saved vector store on Drive.**

Each chunk is converted into a 384-dimensional vector using a local sentence-transformers model. Semantically similar text ends up close together in vector space.



FAISS stores all vectors and finds the nearest ones to a query in milliseconds.

In [ ]:
import os
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading embedding model (downloads ~80MB on first run)...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Using device: {device}")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)
print("Embedding model loaded!")

if os.path.exists(VECTORSTORE_PATH):
    print("\nFound saved vector store on Drive — loading it...")
    vectorstore = FAISS.load_local(
        VECTORSTORE_PATH,
        embedding_model,
        allow_dangerous_deserialization=True
    )
    print(f"Loaded! {vectorstore.index.ntotal} vectors ready.")
else:
    print(f"\nBuilding vector store from {len(chunks)} chunks...")
    vectorstore = FAISS.from_documents(chunks, embedding_model)
    print(f"Built! {vectorstore.index.ntotal} vectors indexed.")
    vectorstore.save_local(VECTORSTORE_PATH)
    print(f"Saved to Drive: {VECTORSTORE_PATH}")

print("\nSanity check — searching for 'what is backpropagation':")
test_results = vectorstore.similarity_search("what is backpropagation", k=2)
for i, doc in enumerate(test_results):
    print(f"  Result {i+1}: [{doc.metadata['title']}] {doc.page_content[:120]}...")

Loading embedding model (downloads ~80MB on first run)...
  Using device: cpu


/tmp/ipykernel_14980/2924431090.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!

Building vector store from 1153 chunks...
Built! 1153 vectors indexed.
Saved to Drive: /content/drive/MyDrive/rag_chatbot/vectorstore/faiss_index

Sanity check — searching for 'what is backpropagation':
  Result 1: [Artificial intelligence] Learning algorithms for neural networks use local search to choose the weights that will get the right output for each i...
  Result 2: [Machine learning] . This line, too, was continued outside the AI/CS field, as "connectionism", by researchers from other disciplines, incl...


---
## Section 5 — Set Up Google Gemini

In [ ]:
import getpass
import os

print("Get your free API key at: https://aistudio.google.com/app/apikey")
print()
GEMINI_API_KEY = getpass.getpass("Paste your Gemini API key here: ")

if len(GEMINI_API_KEY) < 20:
    print("That key looks too short. Please double-check and re-run.")
else:
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
    print("API key stored! (hidden for security)")

Get your free API key at: https://aistudio.google.com/app/apikey

Paste your Gemini API key here: ··········
API key stored! (hidden for security)


---
## Section 6 — Build the RAG Chain with Memory

The chain has four parts:

**LLM** — Gemini 2.5 Flash. Takes a prompt, returns an answer.

**Retriever** — wraps the FAISS vector store. Given a question, returns the top-4 most relevant chunks.

**History-aware retriever** — rewrites follow-up questions into standalone ones using chat history. So "Who invented it?" becomes "Who invented the Transformer architecture?" before hitting the retriever.

**RAG chain** — wires everything together:
1. Rewrite question using history
2. Retrieve top-4 chunks
3. Build prompt: system + context + history + question
4. Call Gemini → get answer
5. Save turn to `chat_history`

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,
)
print("LLM (gemini-2.5-flash) configured")

# Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
print("Retriever configured (top-4 chunks per query)")

# Chat history — plain list of HumanMessage / AIMessage objects
chat_history = []
print("Memory initialized (keeps last 5 turns)")

# Prompt 1: rewrite the user question as standalone using history
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, "
     "rewrite it as a standalone question. "
     "Do NOT answer it. Just reformulate if needed, otherwise return it as is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_prompt)

# Prompt 2: answer using retrieved context
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful AI assistant. Answer using ONLY the context below. "
     "If the answer is not in the context, say you don't know.\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Full RAG chain
qa_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

print("\nRAG chain assembled!")
print("  User question → history-aware retriever → FAISS → Gemini → answer")

LLM (gemini-2.5-flash) configured
Retriever configured (top-4 chunks per query)
Memory initialized (keeps last 5 turns)

RAG chain assembled!
  User question → history-aware retriever → FAISS → Gemini → answer


---
## Section 7 — Test the Chain

Three tests:
- Test 1: direct factual question
- Test 2: follow-up that requires memory to answer correctly
- Test 3: question outside the knowledge base (should not hallucinate)

In [ ]:
def ask(question: str):
    global chat_history

    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print('='*60)

    result = qa_chain.invoke({
        "input": question,
        "chat_history": chat_history,
    })

    answer = result["answer"]
    print(f"Answer:\n{answer}")

    # Save turn to memory
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    # Keep only last 5 turns (10 messages)
    if len(chat_history) > 10:
        chat_history = chat_history[-10:]

    source_titles = list({
        doc.metadata.get("title", "Unknown")
        for doc in result["context"]
    })
    print(f"\nSources : {', '.join(source_titles)}")
    print(f"Chunks  : {len(result['context'])}")


ask("What is a transformer in deep learning?")
ask("When was it introduced and by whom?")
ask("What is the population of Tokyo?")


Question: What is a transformer in deep learning?
Answer:
In deep learning, the transformer is a family of artificial neural network architectures based on the multi-head attention mechanism, in which text is converted to numerical representations called tokens, and each token is converted into a vector via lookup from a word embedding table.

Sources : Artificial intelligence, Transformer (deep learning)
Chunks  : 4

Question: When was it introduced and by whom?
Answer:
The original version of the transformer architecture was proposed in 2017 in the paper "Attention Is All You Need" by researchers at Google.

Sources : Transformer (deep learning)
Chunks  : 4

Question: What is the population of Tokyo?
Answer:
I don't know.

Sources : Artificial intelligence, Machine learning
Chunks  : 4


In [ ]:
# Memory inspection — see what context is currently in chat_history

print(f"Current memory — {len(chat_history)} messages stored:")
print()
for i, msg in enumerate(chat_history):
    role = "User" if isinstance(msg, HumanMessage) else "Bot"
    preview = msg.content[:150].replace("\n", " ")
    print(f"  [{i+1}] {role}: {preview}...")

print()
print("Memory keeps the last 5 exchanges (10 messages).")
print("Older messages are dropped automatically.")

Current memory — 6 messages stored:

  [1] User: What is a transformer in deep learning?...
  [2] Bot: In deep learning, the transformer is a family of artificial neural network architectures based on the multi-head attention mechanism, in which text is...
  [3] User: When was it introduced and by whom?...
  [4] Bot: The original version of the transformer architecture was proposed in 2017 in the paper "Attention Is All You Need" by researchers at Google....
  [5] User: What is the population of Tokyo?...
  [6] Bot: I don't know....

Memory keeps the last 5 exchanges (10 messages).
Older messages are dropped automatically.


---
## Section 8 — Build & Launch the Streamlit UI

Streamlit reruns the entire `app.py` script on every user interaction, so regular variables reset each time. We use:
- `st.session_state` to persist messages and chat history across reruns
- `@st.cache_resource` to load the model and chain only once per session

ngrok creates a public tunnel from `https://<id>.ngrok.io` → `localhost:8501` so you can access the app from your browser.

In [ ]:
app_code = '''
# app.py — RAG Chatbot Streamlit UI
import os
import torch
import streamlit as st
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

st.set_page_config(page_title="RAG Chatbot", page_icon="", layout="centered")

with st.sidebar:
    st.title("Settings")
    st.markdown("**Knowledge base:**")
    st.markdown("- Artificial Intelligence\\n- Machine Learning\\n- NLP\\n- Transformers\\n- LLMs\\n- RAG")
    st.divider()
    api_key = st.text_input("Gemini API Key", type="password", help="Get yours free at aistudio.google.com")
    st.divider()
    if st.button("Clear conversation"):
        st.session_state.messages = []
        st.session_state.chat_history = []
        st.success("Conversation cleared!")
    st.caption("Built with LangChain + FAISS + Gemini")

st.title("RAG Chatbot")
st.caption("Ask me anything about AI and Machine Learning")

if not api_key:
    st.info("Enter your Gemini API key in the sidebar to start")
    st.stop()

@st.cache_resource
def load_chain(api_key: str):
    PROJECT_DIR = "/content/drive/MyDrive/rag_chatbot"
    FAISS_PATH  = f"{PROJECT_DIR}/vectorstore/faiss_index"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    embed = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True}
    )
    vs = FAISS.load_local(FAISS_PATH, embed, allow_dangerous_deserialization=True)

    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key, temperature=0.3)

    contextualize_prompt = ChatPromptTemplate.from_messages([
        ("system", "Rewrite the question as standalone given the history. Do not answer it."),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    history_aware_retriever = create_history_aware_retriever(
        llm, vs.as_retriever(search_kwargs={"k": 4}), contextualize_prompt
    )

    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using ONLY the context below. Say you don\'t know if it\'s not there.\\n\\nContext:\\n{context}"),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    chain = create_retrieval_chain(
        history_aware_retriever,
        create_stuff_documents_chain(llm, qa_prompt)
    )
    return chain

with st.spinner("Loading model and vector store..."):
    chain = load_chain(api_key)

if "messages" not in st.session_state:
    st.session_state.messages = []
    st.session_state.chat_history = []
    st.session_state.messages.append({
        "role": "assistant",
        "content": "Hi! I\'m a RAG chatbot with knowledge about AI and Machine Learning. Ask me anything!",
        "sources": []
    })

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg.get("sources"):
            with st.expander(f"{len(msg[\x27sources\x27])} source(s) used"):
                for source in msg["sources"]:
                    st.caption(f"• {source}")

if user_input := st.chat_input("Ask about AI, ML, NLP, Transformers..."):
    st.session_state.messages.append({"role": "user", "content": user_input, "sources": []})
    with st.chat_message("user"):
        st.markdown(user_input)

    with st.chat_message("assistant"):
        with st.spinner("Searching knowledge base..."):
            result = chain.invoke({"input": user_input, "chat_history": st.session_state.chat_history})

        answer = result["answer"]
        sources = list({doc.metadata.get("title", "Unknown") for doc in result["context"]})

        st.markdown(answer)
        if sources:
            with st.expander(f"{len(sources)} source(s) used"):
                for source in sources:
                    st.caption(f"• {source}")

    st.session_state.messages.append({"role": "assistant", "content": answer, "sources": sources})
    st.session_state.chat_history.append(HumanMessage(content=user_input))
    st.session_state.chat_history.append(AIMessage(content=answer))

    if len(st.session_state.chat_history) > 10:
        st.session_state.chat_history = st.session_state.chat_history[-10:]
'''

with open(APP_PATH, "w") as f:
    f.write(app_code)

print(f"app.py saved to: {APP_PATH}")

app.py saved to: /content/drive/MyDrive/rag_chatbot/app.py


In [ ]:
import os
import time
import getpass
import subprocess
from pyngrok import ngrok, conf

NGROK_TOKEN = getpass.getpass("Paste your ngrok token: ")

if not NGROK_TOKEN or len(NGROK_TOKEN) < 10:
    print("Token looks invalid. Get it at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    conf.get_default().auth_token = NGROK_TOKEN

    os.system("pkill -f streamlit 2>/dev/null")
    time.sleep(2)
    ngrok.kill()
    time.sleep(1)

    process = subprocess.Popen([
        "streamlit", "run", APP_PATH,
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.fileWatcherType", "none"
    ])

    print("Starting Streamlit...")
    time.sleep(5)

    tunnel = ngrok.connect(8501, proto="http")
    public_url = tunnel.public_url

    print(f"\n{'='*50}")
    print(f"Chatbot is LIVE!")
    print(f"{'='*50}")
    print(f"\nOpen: {public_url}")
    print(f"\nEnter your Gemini API key in the sidebar.")
    print(f"Link stays active as long as this Colab tab is open.")

Paste your ngrok token: ··········
Starting Streamlit...

Chatbot is LIVE!

Open: https://unencroaching-vaporously-vanita.ngrok-free.dev

Enter your Gemini API key in the sidebar.
Link stays active as long as this Colab tab is open.


---
## Section 9 — Quick Restart

After a Colab session reset, run only these cells — the vector store is already saved on Drive.

| Cell | What it does | Time |
|------|-------------|------|
| Section 1 | pip install | ~2 min |
| Section 2 | Mount Drive | ~10 sec |
| Section 4 | Load saved FAISS | ~30 sec |
| Section 5 | Enter API key | instant |
| Section 6 | Build chain | ~5 sec |
| Section 8 (launch cell) | Launch app | ~10 sec |

In [ ]:
# Quick restart — run after pip install and Drive mount
import os
import torch
import getpass
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

PROJECT_DIR      = "/content/drive/MyDrive/rag_chatbot"
VECTORSTORE_PATH = f"{PROJECT_DIR}/vectorstore/faiss_index"
APP_PATH         = f"{PROJECT_DIR}/app.py"

if not os.path.exists("/content/drive/MyDrive"):
    print("Drive not mounted — run Section 2 first.")
elif not os.path.exists(VECTORSTORE_PATH):
    print("Vector store not found — run Sections 3 & 4 first.")
else:
    print("Loading embedding model...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True}
    )
    vectorstore = FAISS.load_local(VECTORSTORE_PATH, embedding_model, allow_dangerous_deserialization=True)
    print(f"Vector store loaded ({vectorstore.index.ntotal} vectors)")

    GEMINI_API_KEY = getpass.getpass("\nEnter Gemini API key: ")
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GEMINI_API_KEY, temperature=0.3)

    chat_history = []

    contextualize_prompt = ChatPromptTemplate.from_messages([
        ("system", "Rewrite the question as standalone given the history. Do not answer it."),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    history_aware_retriever = create_history_aware_retriever(
        llm, vectorstore.as_retriever(search_kwargs={"k": 4}), contextualize_prompt
    )
    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using ONLY the context below. Say you don't know if it's not there.\n\nContext:\n{context}"),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    qa_chain = create_retrieval_chain(
        history_aware_retriever,
        create_stuff_documents_chain(llm, qa_prompt)
    )
    print("Chain ready — run the launch cell in Section 8.")

---
## Section 10 — Debugging Tools

Use these cells to inspect what's happening inside the pipeline when something isn't working as expected.

In [ ]:
# Debug: see which chunks the retriever fetches for a given question
# Useful when the bot gives wrong answers — bypasses the LLM entirely

def debug_retrieval(question: str):
    print(f"Question: '{question}'")
    print("─" * 60)

    retrieved_chunks = retriever.invoke(question)

    for i, chunk in enumerate(retrieved_chunks):
        print(f"\nChunk {i+1}/{len(retrieved_chunks)}:")
        print(f"  Source : {chunk.metadata.get('title', 'Unknown')}")
        print(f"  Length : {len(chunk.page_content)} chars")
        for line in chunk.page_content.split("\n")[:5]:
            if line.strip():
                print(f"    {line[:100]}")

debug_retrieval("how does attention mechanism work in transformers?")

Question: 'how does attention mechanism work in transformers?'
────────────────────────────────────────────────────────────

Chunk 1/4:
  Source : Transformer (deep learning)
  Length : 473 chars
    The attention mechanism used in the transformer architecture are scaled dot-product attention units.

Chunk 2/4:
  Source : Transformer (deep learning)
  Length : 272 chars
    {\displaystyle O(N)}
    .
    Ordinary transformers require a memory size that is quadratic in the size of the context window. Att

Chunk 3/4:
  Source : Transformer (deep learning)
  Length : 469 chars
    , determines how the attended tokens influence what information is passed to subsequent layers and u

Chunk 4/4:
  Source : Transformer (deep learning)
  Length : 454 chars
    {\displaystyle c_{j}}
    . This allows the transformer to take any encoded position and find a linear sum of the encoded loca


In [ ]:
# Debug: visualize embedding vectors and cosine similarity
# Shows why semantically similar sentences have similar vectors

import numpy as np

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sentences = [
    "How do neural networks learn?",
    "What is backpropagation in deep learning?",
    "What is the capital city of France?"
]

print("Generating embeddings...\n")
vectors = embedding_model.embed_documents(sentences)

for i, (sent, vec) in enumerate(zip(sentences, vectors)):
    print(f"Sentence {i+1}: '{sent}'")
    print(f"  Dimensions : {len(vec)}")
    print(f"  First 5 vals: {[round(v, 4) for v in vec[:5]]}")
    print()

s12 = cosine_similarity(vectors[0], vectors[1])
s13 = cosine_similarity(vectors[0], vectors[2])
s23 = cosine_similarity(vectors[1], vectors[2])

print("Cosine similarity (1.0 = identical, 0.0 = unrelated):")
print(f"  Sent 1 vs Sent 2 (related)  : {s12:.4f}  — should be HIGH")
print(f"  Sent 1 vs Sent 3 (unrelated): {s13:.4f}  — should be LOW")
print(f"  Sent 2 vs Sent 3 (unrelated): {s23:.4f}  — should be LOW")

Generating embeddings...

Sentence 1: 'How do neural networks learn?'
  Dimensions : 384
  First 5 vals: [-0.0097, -0.1, 0.0047, 0.0257, -0.0027]

Sentence 2: 'What is backpropagation in deep learning?'
  Dimensions : 384
  First 5 vals: [-0.0583, -0.0704, -0.0168, 0.0403, 0.0054]

Sentence 3: 'What is the capital city of France?'
  Dimensions : 384
  First 5 vals: [0.0717, 0.0416, -0.0118, 0.0043, 0.032]

Cosine similarity (1.0 = identical, 0.0 = unrelated):
  Sent 1 vs Sent 2 (related)  : 0.5984  — should be HIGH
  Sent 1 vs Sent 3 (unrelated): 0.0962  — should be LOW
  Sent 2 vs Sent 3 (unrelated): 0.0455  — should be LOW


---
## Section 11 — Extending the Knowledge Base

 can add own documents without rebuilding from scratch. Any text — notes, a PDF chapter, a paper summary — can be chunked and added incrementally to the existing vector store.

In [ ]:
from langchain_core.documents import Document

def add_custom_text(text: str, title: str, source: str = "custom"):
    """Add any text to the existing vector store incrementally."""
    doc = Document(page_content=text, metadata={"title": title, "source": source})
    new_chunks = splitter.split_documents([doc])
    vectorstore.add_documents(new_chunks)
    vectorstore.save_local(VECTORSTORE_PATH)
    print(f"Added '{title}' ({len(new_chunks)} chunks) — total vectors: {vectorstore.index.ntotal}")


# Example — uncomment to test:
add_custom_text(
    text="""
    Attention Is All You Need is a 2017 paper by Vaswani et al.
    It introduced the Transformer architecture, which replaced recurrent
    networks with self-attention, enabling parallel training and becoming
    the foundation for GPT, BERT, Claude, and Gemini.
    """,
    title="Attention Is All You Need (summary)",
    source="custom/vaswani2017"
)

print("Uncomment the example above to add custom text to the knowledge base.")

Added 'Attention Is All You Need (summary)' (1 chunks) — total vectors: 1154
Uncomment the example above to add custom text to the knowledge base.


---
## Summary

### Architecture
```
Wikipedia articles
    ↓  RecursiveCharacterTextSplitter (500 chars, 50 overlap)
~1100 chunks
    ↓  sentence-transformers/all-MiniLM-L6-v2
~1100 vectors (384 dimensions each)
    ↓  FAISS  →  saved to Google Drive

At query time:
User question
    → history-aware retriever (rewrites follow-ups as standalone)
    → FAISS similarity search (top-4 chunks)
    → Gemini 2.5 Flash (answer grounded in context)
    → chat_history list (last 5 turns kept for next query)
```

### Key design decisions
- **`langchain-classic`** used for `create_retrieval_chain` and helpers (moved out of `langchain` core in v1.x)
- **`chat_history` list** of `HumanMessage` / `AIMessage` objects replaces the deprecated `ConversationBufferWindowMemory`
- **`gemini-2.5-flash`** replaces the retired `gemini-1.5-flash`
- Chain invoke uses `"input"` key (not `"question"`) and result sources are in `result["context"]` (not `result["source_documents"]`)

### Skills covered
| Skill | Section |
|---|---|
| Document loading & chunking | 3 |
| Text embeddings & vector search | 4 |
| RAG pipeline | 6 |
| Conversation memory | 6–7 |
| LLM integration (Gemini) | 6–7 |
| Streamlit UI | 8 |
| Google Drive persistence | 2, 4 |

### Resources
- LangChain: https://python.langchain.com
- Sentence Transformers: https://sbert.net
- Streamlit: https://docs.streamlit.io
- FAISS: https://github.com/facebookresearch/faiss